# 11c. Causal Inference: Train Masked Toy Recommender

**Role of this notebook.** Train the leakage-free toy recommender used later to construct the causal treatment. This notebook masks all causal test transition sessions, retrains the four-head MLP recommender on the remaining real interactions, re-estimates value-model head weights without masked sessions, and saves reusable artifacts.

**Steps.**

1. Load notebook-11b test transitions.
2. Build the masked session set containing both session `t` and session `t+1` for every test transition.
3. Load real processed interaction rows and remove all masked sessions.
4. Train a four-head edge MLP for `y_complete`, `y_long`, `y_rewatch`, and `y_neg`, using early stopping on validation BCE.
5. Re-estimate head weights on non-masked validation sessions using an observed-order Plackett-Luce objective over predicted heads.
6. Save masked sessions, model checkpoint, preprocessing object, feature list, head weights, and diagnostics.

This notebook does **not** score test-session candidate videos and does **not** compute treatment variables. Notebook 11d does that.

Default execution is a smoke test. Set `RUN_FULL_TRAINING = True` to write the production artifacts. In full mode, the MLP trains for up to `FULL_EPOCHS = 50` epochs and stops after `EARLY_STOPPING_PATIENCE = 5` validation checks without improvement.


## Feature Policy

The toy MLP feature set must be constructible for every candidate video in notebook 11d.

| Kept | Dropped |
|---|---|
| user static features | author-history features requiring per-candidate lookup |
| video static features | position/rank features such as `sess_rank` |
| level-1/2/3 category ids | realized outcomes such as `watch_ratio` and `play_duration` |
| context known before session | final session length |
| shifted user-level history | any feature from session `t+1` |
| shifted category-level completion history by candidate category | masked test sessions |

The same feature columns and preprocessing object are saved for 11d candidate scoring.


In [1]:
from pathlib import Path
import copy
import json
import math
import random

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import minimize
from scipy.special import logsumexp, softmax
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)

ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
DATA_DIR = ROOT / 'KuaiRec 2.0' / 'data' / 'processed'
OUT_DIR = ROOT / 'python_code_new' / 'outputs'

PREDICTIONS_PATH = OUT_DIR / 'causal_delta_tau_predictions.parquet'
PROCESSED_DATA_PATH = DATA_DIR / 'processed_data.parquet'

MASKED_SESSIONS_PATH = OUT_DIR / 'causal_masked_test_sessions.parquet'
MODEL_PATH = OUT_DIR / 'causal_masked_toy_mlp.pt'
PREPROCESS_PATH = OUT_DIR / 'causal_masked_toy_mlp_preprocess.joblib'
FEATURES_PATH = OUT_DIR / 'causal_masked_toy_mlp_features.json'
HEAD_WEIGHTS_PATH = OUT_DIR / 'causal_masked_head_weights.json'
METRICS_PATH = OUT_DIR / 'causal_masked_toy_recsys_metrics.json'

SMOKE_MASKED_SESSIONS_PATH = OUT_DIR / 'causal_masked_test_sessions_smoke.parquet'
SMOKE_MODEL_PATH = OUT_DIR / 'causal_masked_toy_mlp_smoke.pt'
SMOKE_PREPROCESS_PATH = OUT_DIR / 'causal_masked_toy_mlp_preprocess_smoke.joblib'
SMOKE_FEATURES_PATH = OUT_DIR / 'causal_masked_toy_mlp_features_smoke.json'
SMOKE_HEAD_WEIGHTS_PATH = OUT_DIR / 'causal_masked_head_weights_smoke.json'
SMOKE_METRICS_PATH = OUT_DIR / 'causal_masked_toy_recsys_metrics_smoke.json'

RUN_FULL_TRAINING = False
SMOKE_TRAIN_MAX_ROWS = 80_000
FULL_TRAIN_MAX_ROWS = None
SMOKE_EPOCHS = 1
FULL_EPOCHS = 50
VALID_SHARE = 0.10
TRAIN_BATCH_SIZE = 4_096
EVAL_BATCH_SIZE = 65_536
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-5
DROPOUT = 0.10
HIDDEN_SIZES = [256, 128, 64]
PL_MAX_SESSIONS_SMOKE = 200
PL_MAX_SESSIONS_FULL = 5_000
PL_MAX_ITEMS_PER_SESSION = 80
PL_L2 = 1e-4
RANDOM_SEED = 20260630

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

if RUN_FULL_TRAINING:
    masked_sessions_path = MASKED_SESSIONS_PATH
    model_path = MODEL_PATH
    preprocess_path = PREPROCESS_PATH
    features_path = FEATURES_PATH
    head_weights_path = HEAD_WEIGHTS_PATH
    metrics_path = METRICS_PATH
    train_max_rows = FULL_TRAIN_MAX_ROWS
    num_epochs = FULL_EPOCHS
    pl_max_sessions = PL_MAX_SESSIONS_FULL
else:
    masked_sessions_path = SMOKE_MASKED_SESSIONS_PATH
    model_path = SMOKE_MODEL_PATH
    preprocess_path = SMOKE_PREPROCESS_PATH
    features_path = SMOKE_FEATURES_PATH
    head_weights_path = SMOKE_HEAD_WEIGHTS_PATH
    metrics_path = SMOKE_METRICS_PATH
    train_max_rows = SMOKE_TRAIN_MAX_ROWS
    num_epochs = SMOKE_EPOCHS
    pl_max_sessions = PL_MAX_SESSIONS_SMOKE

for p in [PREDICTIONS_PATH, PROCESSED_DATA_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print('device:', DEVICE)
print('mode:', 'FULL' if RUN_FULL_TRAINING else 'SMOKE')


device: mps
mode: SMOKE


## 1. Load Test Transitions and Mask Sessions

For every test transition,

$$
(user_i, burst_b, session_t) \rightarrow (user_i, burst_b, session_{t+1}),
$$

both sessions are excluded from toy recommender training and head-weight estimation.


In [2]:
pred = pd.read_parquet(PREDICTIONS_PATH)
pred['burst_id'] = pred['burst_id'].astype(int)
test_transitions = pred[pred['split'].eq('test')].copy()
if test_transitions.empty:
    raise ValueError('No test transitions found in causal_delta_tau_predictions.parquet')

mask_t = test_transitions[['user_id', 'burst_id', 'session']].copy()
mask_t1 = test_transitions[['user_id', 'burst_id', 'next_session']].rename(columns={'next_session': 'session'}).copy()
masked_sessions = pd.concat([mask_t, mask_t1], ignore_index=True).drop_duplicates()
masked_sessions['is_masked_session'] = True
masked_sessions.to_parquet(masked_sessions_path, index=False)

print('test transitions:', f'{len(test_transitions):,}')
print('masked sessions:', f'{len(masked_sessions):,}')
display(masked_sessions.head())


test transitions: 4,069
masked sessions: 5,640


,user_id,burst_id,session,is_masked_session
0,3,1,4,True
1,3,1,5,True
2,3,1,6,True
3,3,1,7,True
4,3,1,8,True


## 2. Load Non-Masked Training Rows

The processed real interaction table supplies labels and serving-safe covariates. The notebook removes all masked sessions first, then later creates a session-level train/validation split. The validation side keeps complete sessions so the Plackett-Luce head-weight objective sees the observed order within each held-out session.


In [3]:
continuous_cols = [
    'u_follow_user_num_log1p', 'u_fans_user_num_log1p', 'u_friend_user_num_log1p', 'u_register_days_log1p',
    'i_video_duration', 'i_aspect_ratio',
    'ctx_hour_sin', 'ctx_hour_cos',
    'hist_ema_y_complete', 'hist_ema_y_long', 'hist_ema_y_rewatch', 'hist_ema_y_neg', 'hist_ema_watchratio',
    'hist_cat_ema_complete', 'hist_cat_entropy_l2', 'hist_prev_sess_len', 'hist_intersess_gap_h',
]
binary_cols = ['u_is_lowactive_period', 'u_is_live_streamer', 'u_is_video_author', 'ctx_is_weekend']
categorical_cols = [
    'burst_id',
    'u_user_active_degree', 'u_follow_user_num_range', 'u_fans_user_num_range', 'u_friend_user_num_range', 'u_register_days_range',
] + [f'u_onehot_feat{i}' for i in range(18)] + [
    'i_author_id', 'i_video_type', 'i_upload_type', 'i_visible_status', 'i_music_id', 'i_video_tag_id', 'i_video_tag_name',
    'i_cat_level1_id', 'i_cat_level2_id', 'i_cat_level3_id',
]
label_cols = ['y_complete', 'y_long', 'y_rewatch', 'y_neg']
key_cols = ['user_id', 'video_id', 'burst_id', 'session', 'time']
read_cols = sorted(set(key_cols + continuous_cols + binary_cols + categorical_cols + label_cols))

processed_schema_cols = pd.read_parquet(PROCESSED_DATA_PATH).columns.tolist()
missing_cols = [c for c in read_cols if c not in processed_schema_cols]
if missing_cols:
    raise KeyError(f'processed_data is missing required columns: {missing_cols}')

processed = pd.read_parquet(PROCESSED_DATA_PATH, columns=read_cols)
processed['burst_id'] = processed['burst_id'].astype(int)
processed['session'] = processed['session'].astype(int)
processed = processed.merge(masked_sessions, on=['user_id', 'burst_id', 'session'], how='left')
nonmasked_rows = processed[processed['is_masked_session'].isna()].drop(columns=['is_masked_session']).copy()
masked_row_count = int(processed['is_masked_session'].notna().sum())

nonmasked_rows = (
    nonmasked_rows
    .dropna(subset=label_cols)
    .sort_values(['user_id', 'burst_id', 'session', 'time', 'video_id'], kind='mergesort')
    .reset_index(drop=True)
)

print('processed rows:', f'{len(processed):,}')
print('masked rows removed:', f'{masked_row_count:,}')
print('non-masked usable rows:', f'{len(nonmasked_rows):,}')
print('non-masked usable sessions:', f"{nonmasked_rows[['user_id', 'burst_id', 'session']].drop_duplicates().shape[0]:,}")
display(nonmasked_rows[label_cols].mean().to_frame('label_rate').T)


processed rows: 12,527,912
masked rows removed: 655,461
non-masked usable rows: 11,872,451


non-masked usable sessions: 549,951


,y_complete,y_long,y_rewatch,y_neg
label_rate,0.338483,0.217418,0.076171,0.126399


## 3. Preprocessing and Four-Head MLP

The model predicts four behavior heads from shared edge features:

$$
\widehat y_{ij}=(\widehat p_{complete},\widehat p_{long},\widehat p_{rewatch},\widehat p_{neg}).
$$

The progress bar reports epoch-level training progress.


In [4]:
def fit_preprocess(frame, continuous_cols, binary_cols, categorical_cols):
    cont_median = frame[continuous_cols].apply(pd.to_numeric, errors='coerce').median().fillna(0.0)
    cont_filled = frame[continuous_cols].apply(pd.to_numeric, errors='coerce').fillna(cont_median)
    cont_mean = cont_filled.mean()
    cont_std = cont_filled.std(ddof=0).replace(0.0, 1.0).fillna(1.0)
    bin_median = frame[binary_cols].apply(pd.to_numeric, errors='coerce').median().fillna(0.0)
    cat_maps = {}
    cat_cardinalities = []
    for col in categorical_cols:
        vals = frame[col].astype('object').where(frame[col].notna(), '__MISSING__').astype(str)
        uniques = pd.Index(vals.unique()).sort_values()
        mapping = {str(v): i + 1 for i, v in enumerate(uniques)}
        cat_maps[col] = mapping
        cat_cardinalities.append(len(mapping) + 1)
    return {
        'continuous_cols': continuous_cols,
        'binary_cols': binary_cols,
        'categorical_cols': categorical_cols,
        'cont_median': cont_median.to_dict(),
        'cont_mean': cont_mean.to_dict(),
        'cont_std': cont_std.to_dict(),
        'bin_median': bin_median.to_dict(),
        'cat_maps': cat_maps,
        'cat_cardinalities': cat_cardinalities,
    }


def transform_frame(frame, prep):
    cont_cols = prep['continuous_cols']
    bin_cols = prep['binary_cols']
    cat_cols = prep['categorical_cols']
    cont = frame[cont_cols].apply(pd.to_numeric, errors='coerce')
    for col in cont_cols:
        cont[col] = cont[col].fillna(prep['cont_median'][col])
        cont[col] = (cont[col] - prep['cont_mean'][col]) / prep['cont_std'][col]
    bins = frame[bin_cols].apply(pd.to_numeric, errors='coerce')
    for col in bin_cols:
        bins[col] = bins[col].fillna(prep['bin_median'][col])
    x_cat = np.zeros((len(frame), len(cat_cols)), dtype=np.int64)
    for j, col in enumerate(cat_cols):
        vals = frame[col].astype('object').where(frame[col].notna(), '__MISSING__').astype(str)
        x_cat[:, j] = vals.map(prep['cat_maps'][col]).fillna(0).to_numpy(dtype=np.int64)
    return cont.to_numpy(dtype=np.float32), bins.to_numpy(dtype=np.float32), x_cat

class EdgeMLP(nn.Module):
    def __init__(self, n_cont, n_bin, cat_cardinalities, hidden_sizes, dropout=0.1, emb_dim_cap=32):
        super().__init__()
        self.embeddings = nn.ModuleList()
        emb_dims = []
        for card in cat_cardinalities:
            dim = min(emb_dim_cap, max(4, int(round(card ** 0.25 * 8))))
            self.embeddings.append(nn.Embedding(int(card), dim))
            emb_dims.append(dim)
        input_dim = int(n_cont) + int(n_bin) + int(sum(emb_dims))
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, 4))
        self.net = nn.Sequential(*layers)

    def forward(self, x_cont, x_bin, x_cat):
        parts = [x_cont, x_bin]
        parts.extend([emb(x_cat[:, j]) for j, emb in enumerate(self.embeddings)])
        return self.net(torch.cat(parts, dim=1))


def make_loader(x_cont, x_bin, x_cat, y=None, batch_size=TRAIN_BATCH_SIZE, shuffle=False):
    tensors = [torch.from_numpy(x_cont), torch.from_numpy(x_bin), torch.from_numpy(x_cat)]
    if y is not None:
        tensors.append(torch.from_numpy(y.astype(np.float32)))
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle)

rng = np.random.default_rng(RANDOM_SEED)
session_key_cols = ['user_id', 'burst_id', 'session']
session_keys = nonmasked_rows[session_key_cols].drop_duplicates().reset_index(drop=True)
perm = rng.permutation(len(session_keys))
n_valid_sessions = max(1, int(round(len(session_keys) * VALID_SHARE)))
valid_session_keys = session_keys.iloc[perm[:n_valid_sessions]].reset_index(drop=True)
train_session_keys = session_keys.iloc[perm[n_valid_sessions:]].reset_index(drop=True)

train_pool = (
    nonmasked_rows
    .merge(train_session_keys.assign(_split_session=1), on=session_key_cols, how='inner')
    .drop(columns=['_split_session'])
    .sort_values(['user_id', 'burst_id', 'session', 'time', 'video_id'], kind='mergesort')
    .reset_index(drop=True)
)
valid_fit = (
    nonmasked_rows
    .merge(valid_session_keys.assign(_split_session=1), on=session_key_cols, how='inner')
    .drop(columns=['_split_session'])
    .sort_values(['user_id', 'burst_id', 'session', 'time', 'video_id'], kind='mergesort')
    .reset_index(drop=True)
)

if train_max_rows is not None and len(train_pool) > int(train_max_rows):
    train_fit = (
        train_pool
        .sample(n=int(train_max_rows), random_state=RANDOM_SEED)
        .sort_values(['user_id', 'burst_id', 'session', 'time', 'video_id'], kind='mergesort')
        .reset_index(drop=True)
    )
else:
    train_fit = train_pool.reset_index(drop=True)
train_rows = train_fit

split_overlap = train_session_keys.merge(valid_session_keys, on=session_key_cols, how='inner')
if len(split_overlap):
    raise AssertionError('Train and validation session splits overlap')
if len(train_fit) == 0 or len(valid_fit) == 0:
    raise ValueError('Empty train or validation split')

print('split unit: session')
print('train sessions:', f'{len(train_session_keys):,}', 'validation sessions:', f'{len(valid_session_keys):,}')
print('train pool rows:', f'{len(train_pool):,}', 'train rows used:', f'{len(train_fit):,}', 'validation rows:', f'{len(valid_fit):,}')

prep = fit_preprocess(train_fit, continuous_cols, binary_cols, categorical_cols)
x_cont_train, x_bin_train, x_cat_train = transform_frame(train_fit, prep)
x_cont_valid, x_bin_valid, x_cat_valid = transform_frame(valid_fit, prep)
y_train = train_fit[label_cols].to_numpy(dtype=np.float32)
y_valid = valid_fit[label_cols].to_numpy(dtype=np.float32)

train_loader = make_loader(x_cont_train, x_bin_train, x_cat_train, y_train, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
valid_loader = make_loader(x_cont_valid, x_bin_valid, x_cat_valid, y_valid, batch_size=EVAL_BATCH_SIZE, shuffle=False)

model = EdgeMLP(len(continuous_cols), len(binary_cols), prep['cat_cardinalities'], HIDDEN_SIZES, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss(reduction='sum')

@torch.no_grad()
def predict_probs_frame(frame, batch_size=EVAL_BATCH_SIZE):
    x_cont, x_bin, x_cat = transform_frame(frame, prep)
    loader = make_loader(x_cont, x_bin, x_cat, y=None, batch_size=batch_size, shuffle=False)
    model.eval()
    out = []
    for batch in loader:
        x_cont_b, x_bin_b, x_cat_b = [b.to(DEVICE) for b in batch]
        out.append(torch.sigmoid(model(x_cont_b, x_bin_b, x_cat_b)).cpu().numpy())
    return np.vstack(out)

@torch.no_grad()
def evaluate_loader(loader):
    model.eval()
    total_loss = 0.0
    total_n = 0
    preds = []
    ys = []
    for batch in loader:
        x_cont, x_bin, x_cat, y = [b.to(DEVICE) for b in batch]
        logits = model(x_cont, x_bin, x_cat)
        loss = criterion(logits, y)
        total_loss += float(loss.item())
        total_n += int(y.numel())
        preds.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.cpu().numpy())
    return total_loss / max(total_n, 1), np.vstack(preds), np.vstack(ys)

history = []
best_valid_loss = float('inf')
best_epoch = 0
best_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0
early_stopped = False

for epoch in tqdm(range(1, num_epochs + 1), desc='Training progress', unit='epoch'):
    model.train()
    train_loss_sum = 0.0
    train_count = 0
    for batch in train_loader:
        x_cont, x_bin, x_cat, y = [b.to(DEVICE) for b in batch]
        optimizer.zero_grad(set_to_none=True)
        logits = model(x_cont, x_bin, x_cat)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss_sum += float(loss.item())
        train_count += int(y.numel())

    valid_loss, valid_pred, valid_y = evaluate_loader(valid_loader)
    train_bce = train_loss_sum / max(train_count, 1)
    improved = valid_loss < (best_valid_loss - EARLY_STOPPING_MIN_DELTA)
    if improved:
        best_valid_loss = float(valid_loss)
        best_epoch = int(epoch)
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    history.append({
        'epoch': int(epoch),
        'train_bce': float(train_bce),
        'valid_bce': float(valid_loss),
        'best_valid_bce': float(best_valid_loss),
        'is_best_epoch': bool(improved),
        'epochs_without_improvement': int(epochs_without_improvement),
    })

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        early_stopped = True
        print(f'Early stopping at epoch {epoch}; best epoch {best_epoch} with valid BCE {best_valid_loss:.6f}.')
        break

model.load_state_dict(best_state)
valid_loss, valid_pred, valid_y = evaluate_loader(valid_loader)
print(f'using best epoch: {best_epoch} with valid BCE: {valid_loss:.6f}')
valid_metrics_by_head = {}
for j, col in enumerate(label_cols):
    y = valid_y[:, j]
    p = valid_pred[:, j]
    valid_metrics_by_head[col] = {
        'auc': None if len(np.unique(y)) < 2 else float(roc_auc_score(y, p)),
        'average_precision': None if len(np.unique(y)) < 2 else float(average_precision_score(y, p)),
    }
display(pd.DataFrame(history))
print('valid BCE:', valid_loss)
print(json.dumps(valid_metrics_by_head, indent=2))


split unit: session
train sessions: 494,956 validation sessions: 54,995
train pool rows: 10,681,219 train rows used: 80,000 validation rows: 1,191,232


Training progress:   0%|          | 0/1 [00:00<?, ?epoch/s]

using best epoch: 1 with valid BCE: 0.454510


,epoch,train_bce,valid_bce,best_valid_bce,is_best_epoch,epochs_without_improvement
0,1,0.505182,0.45451,0.45451,True,0


valid BCE: 0.454510493619998
{
  "y_complete": {
    "auc": 0.6413410033134792,
    "average_precision": 0.4634331224927767
  },
  "y_long": {
    "auc": 0.6099969006653521,
    "average_precision": 0.287753467855204
  },
  "y_rewatch": {
    "auc": 0.6116704054203036,
    "average_precision": 0.1073836761371801
  },
  "y_neg": {
    "auc": 0.5963246311491622,
    "average_precision": 0.16605306739437006
  }
}


## 4. Re-Estimate Head Weights Without Masked Sessions

The score is:

$$
score_{ij}=w_c\widehat p_{complete,ij}+w_l\widehat p_{long,ij}+w_r\widehat p_{rewatch,ij}+w_n\widehat p_{neg,ij}.
$$

Weights are re-estimated on complete non-masked validation sessions. The objective is an observed-order Plackett-Luce likelihood within each validation session. This is leakage-free because no masked session appears in the validation rows used for weight fitting, and it is ranking-valid because validation rows are selected by session rather than by individual interaction.


In [5]:
pl_rows = valid_fit[['user_id', 'burst_id', 'session', 'time', 'video_id'] + label_cols].copy()
pl_probs = valid_pred.astype(np.float64)
for j, name in enumerate(['p_complete_hat', 'p_long_hat', 'p_rewatch_hat', 'p_neg_hat']):
    pl_rows[name] = pl_probs[:, j]

session_keys = pl_rows[['user_id', 'burst_id', 'session']].drop_duplicates()
if len(session_keys) > pl_max_sessions:
    session_keys = session_keys.sample(n=pl_max_sessions, random_state=RANDOM_SEED)
pl_rows = pl_rows.merge(session_keys, on=['user_id', 'burst_id', 'session'], how='inner')
pl_rows = pl_rows.sort_values(['user_id', 'burst_id', 'session', 'time', 'video_id'], kind='mergesort').reset_index(drop=True)

X_sessions = []
for _, g in pl_rows.groupby(['user_id', 'burst_id', 'session'], sort=False):
    X = g[['p_complete_hat', 'p_long_hat', 'p_rewatch_hat', 'p_neg_hat']].to_numpy(dtype=np.float64)
    if len(X) >= 2:
        X_sessions.append(X[:PL_MAX_ITEMS_PER_SESSION])

if not X_sessions:
    raise ValueError('No validation sessions available for PL head-weight estimation')

def pl_loss_grad(w, sessions, l2=PL_L2):
    w = np.asarray(w, dtype=np.float64)
    loss = float(l2) * float(w @ w)
    grad = 2.0 * float(l2) * w
    n_terms = 0
    for X in sessions:
        s = X @ w
        m = len(X)
        for r in range(m - 1):
            rem_s = s[r:]
            rem_X = X[r:]
            probs = softmax(rem_s)
            loss += float(logsumexp(rem_s) - s[r])
            grad += probs @ rem_X - X[r]
            n_terms += 1
    scale = max(n_terms, 1)
    return loss / scale, grad / scale

def fit_pl(sign_constrained=False):
    bounds = [(0.0, None), (0.0, None), (0.0, None), (None, 0.0)] if sign_constrained else None
    def fun(w): return pl_loss_grad(w, X_sessions)[0]
    def jac(w): return pl_loss_grad(w, X_sessions)[1]
    result = minimize(fun, x0=np.array([0.05, 0.02, 0.02, -0.10]), jac=jac, method='L-BFGS-B', bounds=bounds, options={'maxiter': 500})
    return result

fit_unconstrained = fit_pl(sign_constrained=False)
fit_constrained = fit_pl(sign_constrained=True)
PRIMARY_WEIGHT_FIT = 'unconstrained'
primary = fit_unconstrained if PRIMARY_WEIGHT_FIT == 'unconstrained' else fit_constrained
weight_names = ['w_complete', 'w_long', 'w_rewatch', 'w_neg']
head_weights = {name: float(v) for name, v in zip(weight_names, primary.x)}
head_weight_payload = {
    'weights': head_weights,
    'primary_fit': PRIMARY_WEIGHT_FIT,
    'objective_value': float(primary.fun),
    'num_sessions_used': int(len(X_sessions)),
    'num_rows_used': int(sum(len(x) for x in X_sessions)),
    'max_items_per_session': int(PL_MAX_ITEMS_PER_SESSION),
    'constraints_used_for_primary': bool(PRIMARY_WEIGHT_FIT == 'constrained'),
    'normalization': 'none; score is linear in predicted probabilities',
    'unconstrained': {'success': bool(fit_unconstrained.success), 'objective_value': float(fit_unconstrained.fun), 'weights': {name: float(v) for name, v in zip(weight_names, fit_unconstrained.x)}},
    'constrained': {'success': bool(fit_constrained.success), 'objective_value': float(fit_constrained.fun), 'weights': {name: float(v) for name, v in zip(weight_names, fit_constrained.x)}},
    'leakage_rule': 'estimated only on complete non-masked validation sessions',
    'validation_split_unit': 'session',
}
print(json.dumps(head_weight_payload, indent=2))


{
  "weights": {
    "w_complete": -2.325087244035678,
    "w_long": 13.282549930908198,
    "w_rewatch": -6.375461153087586,
    "w_neg": -2.8852823795181486
  },
  "primary_fit": "unconstrained",
  "objective_value": 2.83065195235609,
  "num_sessions_used": 181,
  "num_rows_used": 3566,
  "max_items_per_session": 80,
  "constraints_used_for_primary": false,
  "normalization": "none; score is linear in predicted probabilities",
  "unconstrained": {
    "success": true,
    "objective_value": 2.83065195235609,
    "weights": {
      "w_complete": -2.325087244035678,
      "w_long": 13.282549930908198,
      "w_rewatch": -6.375461153087586,
      "w_neg": -2.8852823795181486
    }
  },
  "constrained": {
    "success": true,
    "objective_value": 2.830904081079415,
    "weights": {
      "w_complete": 0.0,
      "w_long": 9.481645377930695,
      "w_rewatch": 0.0,
      "w_neg": -6.58070075730246
    }
  },
  "leakage_rule": "estimated only on complete non-masked validation sessions",


## 5. Save Artifacts

The output artifacts are the only inputs 11d needs from this notebook.


In [6]:
feature_payload = {
    'continuous_cols': continuous_cols,
    'binary_cols': binary_cols,
    'categorical_cols': categorical_cols,
    'label_cols': label_cols,
    'excluded_features': [
        'hist_author_recency_days', 'hist_has_author_history', 'hist_last_complete_author',
        'sess_rank', 'position_in_session', 'sess_len',
        'play_duration', 'watch_ratio', 'session t+1 variables',
    ],
}

torch.save({
    'state_dict': model.state_dict(),
    'continuous_cols': continuous_cols,
    'binary_cols': binary_cols,
    'categorical_cols': categorical_cols,
    'cat_cardinalities': prep['cat_cardinalities'],
    'hidden_sizes': HIDDEN_SIZES,
    'dropout': DROPOUT,
    'label_cols': label_cols,
}, model_path)
joblib.dump(prep, preprocess_path)
features_path.write_text(json.dumps(feature_payload, indent=2) + '\n')
head_weights_path.write_text(json.dumps(head_weight_payload, indent=2) + '\n')

metrics_payload = {
    'mode': 'full' if RUN_FULL_TRAINING else 'smoke',
    'run_full_training': bool(RUN_FULL_TRAINING),
    'random_seed': RANDOM_SEED,
    'num_test_transitions': int(len(test_transitions)),
    'num_masked_sessions': int(len(masked_sessions)),
    'processed_rows': int(len(processed)),
    'masked_rows_removed': int(masked_row_count),
    'nonmasked_rows_available': int(len(nonmasked_rows)),
    'training_rows_after_masking_and_sampling': int(len(train_rows)),
    'train_rows': int(len(train_fit)),
    'train_pool_rows': int(len(train_pool)),
    'validation_rows': int(len(valid_fit)),
    'split_unit': 'session',
    'train_sessions': int(len(train_session_keys)),
    'validation_sessions': int(len(valid_session_keys)),
    'validation_bce': float(valid_loss),
    'validation_metrics_by_head': valid_metrics_by_head,
    'num_epochs_requested': int(num_epochs),
    'epochs_trained': int(len(history)),
    'early_stopping_patience': int(EARLY_STOPPING_PATIENCE),
    'early_stopping_min_delta': float(EARLY_STOPPING_MIN_DELTA),
    'early_stopped': bool(early_stopped),
    'best_epoch': int(best_epoch),
    'best_validation_bce': float(best_valid_loss),
    'training_history': history,
    'head_weights': head_weight_payload,
    'feature_columns': feature_payload,
    'masked_sessions_excluded_from_mlp_training': True,
    'masked_sessions_excluded_from_head_weight_estimation': True,
    'head_weight_validation_sessions_are_complete': True,
    'output_paths': {
        'masked_sessions': str(masked_sessions_path),
        'model': str(model_path),
        'preprocess': str(preprocess_path),
        'features': str(features_path),
        'head_weights': str(head_weights_path),
        'metrics': str(metrics_path),
    },
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2, allow_nan=True) + '\n')

print('saved masked sessions:', masked_sessions_path)
print('saved model:', model_path)
print('saved preprocess:', preprocess_path)
print('saved features:', features_path)
print('saved head weights:', head_weights_path)
print('saved metrics:', metrics_path)


saved masked sessions: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_masked_test_sessions_smoke.parquet
saved model: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_masked_toy_mlp_smoke.pt
saved preprocess: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_masked_toy_mlp_preprocess_smoke.joblib
saved features: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_masked_toy_mlp_features_smoke.json
saved head weights: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_masked_head_weights_smoke.json
saved metrics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_masked_toy_recsys_metrics_smoke.json
